##### Task 17
Separate bad records from good records.

What I want in the end:
- “Good” tables used for analytics
- “Quarantine” tables for rejected records
- A clear reason column explaining why each rejected record failed validation

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- Possible reasons: invalid customer, invalid date, duplicate order, invalid item reference
CREATE OR REPLACE TEMP VIEW invalid_customer_reference AS
SELECT o.order_id, o.customer_id, o.item_id
FROM (
  SELECT order_id, customer_id, item.item_id
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN bronze_customers c
  ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
GROUP BY o.order_id, o.customer_id, o.item_id;

CREATE OR REPLACE TEMP VIEW invalid_order_dates AS
SELECT order_id, customer_id, item.item_id, order_date
FROM bronze_orders
LATERAL VIEW explode(items) AS item
WHERE try_to_date(order_date, 'yyyy-MM-dd') IS NULL;

CREATE OR REPLACE TEMP VIEW duplicate_orders AS
SELECT order_id, customer_id, item.item_id
FROM (
  SELECT order_id, customer_id, items,
  ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY order_id
  ) rn
  FROM bronze_orders
) t
LATERAL VIEW explode(items) AS item
WHERE rn > 1;

CREATE OR REPLACE TEMP VIEW invalid_item_reference AS
SELECT o.order_id, o.customer_id, o.item_id
FROM (
  SELECT order_id, customer_id, item.item_id
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN bronze_items i
  ON o.item_id = i.item_id
WHERE i.item_id IS NULL;

CREATE OR REPLACE TABLE quarantine_orders AS
SELECT order_id, customer_id, item_id, 'missing_customer' reason FROM invalid_customer_reference
UNION ALL
SELECT order_id, customer_id, item_id, 'invalid_date' FROM invalid_order_dates
UNION ALL
SELECT order_id, customer_id, item_id, 'duplicate_order' FROM duplicate_orders
UNION ALL
SELECT order_id, customer_id, item_id, 'invalid_item_reference' FROM invalid_item_reference
ORDER BY order_id, customer_id, item_id;

SELECT *
FROM quarantine_orders

In [0]:
%sql
-- Possible reasons: missing email, duplicate entry, invalid created_at
CREATE OR REPLACE TEMP VIEW customers_missing_email AS
SELECT customer_id
FROM bronze_customers
WHERE email IS NULL;

CREATE OR REPLACE TEMP VIEW duplicate_customers AS
SELECT customer_id
FROM (
  SELECT customer_id,
  ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY customer_id
  ) rn
  FROM bronze_customers
) t
WHERE rn > 1;

CREATE OR REPLACE TEMP VIEW invalid_date AS
SELECT customer_id, created_at
FROM bronze_customers
WHERE try_to_date(created_at, 'yyyy-MM-dd') IS NULL;

CREATE OR REPLACE TABLE quarantine_customers AS
SELECT customer_id, 'missing_email' reason FROM customers_missing_email
UNION ALL
SELECT customer_id, 'duplicate_entry' FROM duplicate_customers
UNION ALL
SELECT customer_id, 'invalid_date' FROM invalid_date
ORDER BY customer_id, reason;

SELECT *
FROM quarantine_customers;

In [0]:
%sql
-- possible reasons: null price, duplicate entry

CREATE OR REPLACE TEMP VIEW item_price_invalid AS
SELECT item_id, price
FROM bronze_items
WHERE price IS NULL OR price <= 0;

CREATE OR REPLACE TEMP VIEW duplicate_items AS
SELECT item_id
FROM (
  SELECT item_id,
  ROW_NUMBER() OVER (
    PARTITION BY item_id
    ORDER BY item_id
  ) rn
  FROM bronze_items
) t
WHERE rn < 1;

CREATE OR REPLACE TABLE quarantine_items AS
SELECT item_id, 'invalid_price' reason FROM item_price_invalid
UNION ALL
SELECT item_id, 'duplicate_entry' FROM duplicate_items
ORDER BY item_id, reason;

SELECT *
FROM quarantine_items;